# BTC/USD — Backtest + Monte Carlo
**Signal logika iz `signal_tracker.py v1.2`**  
Tocke ≥ +3 → Long | Tocke ≤ −3 → Short | ATR-based SL/TP

```
pip install yfinance numpy scipy matplotlib ipywidgets
jupyter nbextension enable --py widgetsnbextension  # ce widgets ne delajo
```

In [1]:
# ── NAMESTITEV (poženi enkrat) ────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'yfinance', 'numpy', 'scipy', 'matplotlib', 'ipywidgets', '-q'])
print('Paketi OK')

Paketi OK


In [2]:
# ── UVOZI ─────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import yfinance as yf
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 110
print('Uvozi OK')

Uvozi OK


---
## 1 · Indikatorji & signal logika

In [3]:
# ═══════════════════════════════════════════════════════
#  INDIKATORJI  (direktno iz signal_tracker.py v1.2)
# ═══════════════════════════════════════════════════════

def calc_atr(closes, highs, lows, period=14):
    if len(closes) < period + 1:
        return None
    trs = []
    for i in range(1, len(closes)):
        hl  = highs[i] - lows[i]
        hpc = abs(highs[i]  - closes[i-1])
        lpc = abs(lows[i]   - closes[i-1])
        trs.append(max(hl, hpc, lpc))
    atr = float(np.mean(trs[:period]))
    for x in trs[period:]:
        atr = (atr * (period - 1) + x) / period
    return atr

def calc_rsi(closes, period=14):
    if len(closes) < period + 1:
        return None
    diffs  = np.diff(closes)
    gains  = np.where(diffs > 0, diffs, 0.0)
    losses = np.where(diffs < 0, -diffs, 0.0)
    ag = float(np.mean(gains[:period]))
    al = float(np.mean(losses[:period]))
    for g, l in zip(gains[period:], losses[period:]):
        ag = (ag * (period-1) + g) / period
        al = (al * (period-1) + l) / period
    if al == 0:
        return 100.0
    return round(100 - (100 / (1 + ag/al)), 2)

def calc_macd(closes, fast=12, slow=26, signal=9):
    if len(closes) < slow + signal:
        return None, None, False, False
    def ema(data, n):
        k = 2/(n+1)
        e = float(np.mean(data[:n]))
        out = [e]
        for x in data[n:]:
            e = x*k + e*(1-k)
            out.append(e)
        return out
    fe  = ema(closes, fast)
    se  = ema(closes, slow)
    off = slow - fast
    ml  = [f - s for f, s in zip(fe[off:], se)]
    if len(ml) < signal:
        return None, None, False, False
    sl_ = ema(ml, signal)
    m1, s1 = ml[-1], sl_[-1]
    m0 = ml[-2] if len(ml) > 1 else m1
    s0 = sl_[-2] if len(sl_) > 1 else s1
    return m1, s1, (m1 > s1 and m0 <= s0), (m1 < s1 and m0 >= s0)

def rough_heston_vol(closes, H=0.08, window=30):
    if len(closes) < window + 2:
        rets = np.diff(np.log(np.maximum(closes, 1e-10)))
        return float(np.std(rets) * np.sqrt(8760))
    donosi = np.diff(np.log(np.maximum(closes, 1e-10)))
    w = min(window, len(donosi))
    weights = np.array([(w - i) ** (H - 0.5) for i in range(w)])
    weights /= weights.sum()
    local_var = float(np.sum(weights * donosi[-w:] ** 2))
    return min(max(float(np.sqrt(local_var * 8760)), 0.03), 5.0)

def vol_rezim(closes, H=0.08, window=30):
    if len(closes) < window * 2 + 2:
        return 'NORMAL', 1.0
    v1 = rough_heston_vol(closes[-window:],          H, window//2)
    v2 = rough_heston_vol(closes[-window*2:-window], H, window//2)
    if v2 == 0:
        return 'NORMAL', 1.0
    ratio = v1 / v2
    if ratio > 1.4:   return 'EXPANDING',   round(ratio, 3)
    elif ratio < 0.7: return 'CONTRACTING', round(ratio, 3)
    return 'NORMAL', round(ratio, 3)

def izracunaj_tocke(closes, highs, lows, interval='1h'):
    """Vrne (tocke, sigma, atr_korekcija, rh_rezim) — ista logika kot tracker v1.2"""
    n     = len(closes)
    S     = closes[-1]
    tocke = 0

    # 1. SMA200
    sma200 = float(np.mean(closes[-200:])) if n >= 200 else float(np.mean(closes))
    if S > sma200: tocke += 2
    else:          tocke -= 2

    # 2. RSI
    rsi = calc_rsi(closes, 14)
    if rsi is not None:
        if 40 <= rsi <= 65:   tocke += 2
        elif 20 < rsi < 40:   tocke += 1
        elif rsi > 65:        tocke -= 2
        elif rsi <= 20:       tocke -= 1

    # 3. MACD
    mv, sv, cu, cd = calc_macd(closes)
    if mv is not None:
        if cu:          tocke += 2
        elif mv > sv:   tocke += 1
        elif cd:        tocke -= 2
        elif mv < sv:   tocke -= 1

    # 4. Vol penalti
    sigma = rough_heston_vol(closes, H=0.08, window=30)
    if sigma > 0.60:   tocke -= 1
    elif sigma < 0.08: tocke += 1

    # 5. Rough Heston rezim → ATR korekcija
    rh_rezim, _ = vol_rezim(closes, H=0.08, window=30)
    if rh_rezim == 'EXPANDING':
        atr_kor = 1.15
    elif rh_rezim == 'CONTRACTING':
        atr_kor = 0.92 if interval == '1h' else 0.85
    else:
        atr_kor = 1.0

    return tocke, sigma, atr_kor, rh_rezim

print('Indikatorji naloženi ✓')

Indikatorji naloženi ✓


---
## 2 · Backtest engine

In [4]:
# ═══════════════════════════════════════════════════════
#  ATR MULTIPLIKATORJI (iz v1.2)
# ═══════════════════════════════════════════════════════
ATR_MULT_BY_TF = {
    '15m': {'sl': 2.0,  'tp': 4.0},
    '1h':  {'sl': 2.5,  'tp': 5.0},
    '1d':  {'sl': 1.5,  'tp': 3.0},
}

def run_backtest(df, interval='1h', signal_long=3, signal_short=-3,
                 sl_mult=None, tp_mult=None, provizija=0.0006):
    """
    Glavna backtest funkcija.
    sl_mult / tp_mult: ce None, vzame iz ATR_MULT_BY_TF
    """
    closes = df['Close'].values
    highs  = df['High'].values
    lows   = df['Low'].values
    dates  = df.index

    mult = ATR_MULT_BY_TF.get(interval, ATR_MULT_BY_TF['1h'])
    SL_M = sl_mult if sl_mult is not None else mult['sl']
    TP_M = tp_mult if tp_mult is not None else mult['tp']

    WARM   = max(210, 62)  # SMA200 + MACD warm-up
    trades = []
    i      = WARM

    while i < len(closes) - 2:
        c_s = closes[:i+1]
        h_s = highs[:i+1]
        l_s = lows[:i+1]

        tocke, sigma, atr_kor, rh_rezim = izracunaj_tocke(c_s, h_s, l_s, interval)

        if tocke >= signal_long:
            smer = 'LONG'
        elif tocke <= signal_short:
            smer = 'SHORT'
        else:
            i += 1
            continue

        vstop_idx  = i + 1
        vstop_cena = closes[vstop_idx]
        vstop_dat  = dates[vstop_idx]

        atr_raw = calc_atr(c_s, h_s, l_s, 14)
        if atr_raw is None:
            i += 1
            continue
        atr = atr_raw * atr_kor

        if smer == 'LONG':
            sl = vstop_cena - atr * SL_M
            tp = vstop_cena + atr * TP_M
        else:
            sl = vstop_cena + atr * SL_M
            tp = vstop_cena - atr * TP_M

        izhod_cena  = None
        izhod_dat   = None
        izhod_vzrok = 'OPEN'

        for j in range(vstop_idx + 1, len(closes)):
            h, l = highs[j], lows[j]
            if smer == 'LONG':
                if l <= sl:  izhod_cena = sl;  izhod_dat = dates[j]; izhod_vzrok = 'SL'; break
                if h >= tp:  izhod_cena = tp;  izhod_dat = dates[j]; izhod_vzrok = 'TP'; break
            else:
                if h >= sl:  izhod_cena = sl;  izhod_dat = dates[j]; izhod_vzrok = 'SL'; break
                if l <= tp:  izhod_cena = tp;  izhod_dat = dates[j]; izhod_vzrok = 'TP'; break

        if izhod_cena is None:
            izhod_cena  = closes[-1]
            izhod_dat   = dates[-1]
            izhod_vzrok = 'OPEN'

        if smer == 'LONG':
            raw = (izhod_cena - vstop_cena) / vstop_cena
        else:
            raw = (vstop_cena - izhod_cena) / vstop_cena
        pnl_pct = raw - 2 * provizija

        trades.append({
            'vstop_datum':  vstop_dat,
            'izhod_datum':  izhod_dat,
            'smer':         smer,
            'vstop_cena':   vstop_cena,
            'izhod_cena':   izhod_cena,
            'tocke':        tocke,
            'pnl_pct':      pnl_pct * 100,
            'izhod_vzrok':  izhod_vzrok,
            'atr':          atr,
            'sigma':        sigma,
        })

        # Preskoči na svečko po izhodu — preprečuje overlap
        try:
            izhod_idx = list(dates).index(izhod_dat)
        except ValueError:
            izhod_idx = vstop_idx + 1
        i = max(izhod_idx + 1, i + 1)

    return trades


def equity_krivulja(trades, kapital=1000.0):
    kap = kapital
    eq  = [kap]
    for t in trades:
        kap *= (1 + t['pnl_pct'] / 100)
        eq.append(kap)
    return np.array(eq)


def calc_drawdown(equity):
    peak = np.maximum.accumulate(equity)
    return (equity - peak) / peak * 100


def calc_statistike(trades, equity, kapital):
    if not trades:
        return {}
    pnls  = np.array([t['pnl_pct'] for t in trades])
    wins  = pnls[pnls > 0]
    loss  = pnls[pnls <= 0]
    dd    = calc_drawdown(equity)
    rr    = abs(np.mean(wins)) / abs(np.mean(loss)) if len(wins) > 0 and len(loss) > 0 else 0
    sharpe = (np.mean(pnls) / np.std(pnls) * np.sqrt(len(pnls))) if np.std(pnls) > 0 else 0
    return {
        'skupaj_trades':    len(trades),
        'long_trades':      sum(1 for t in trades if t['smer'] == 'LONG'),
        'short_trades':     sum(1 for t in trades if t['smer'] == 'SHORT'),
        'win_rate':         len(wins) / len(pnls) * 100,
        'avg_win':          float(np.mean(wins)) if len(wins) > 0 else 0,
        'avg_loss':         float(np.mean(loss)) if len(loss) > 0 else 0,
        'rr_avg':           rr,
        'sharpe':           sharpe,
        'skupni_donos_pct': (equity[-1] / kapital - 1) * 100,
        'skupni_donos_eur': equity[-1] - kapital,
        'max_dd_pct':       float(np.min(dd)),
        'konc_kapital':     equity[-1],
        'tp_trades':        sum(1 for t in trades if t['izhod_vzrok'] == 'TP'),
        'sl_trades':        sum(1 for t in trades if t['izhod_vzrok'] == 'SL'),
    }


def monte_carlo(trades, n_sim=200, kapital=1000.0, seed=42):
    rng  = np.random.default_rng(seed)
    pnls = np.array([t['pnl_pct'] for t in trades])
    n    = len(pnls)
    mat  = np.zeros((n_sim, n + 1))
    mat[:, 0] = kapital
    for s in range(n_sim):
        sh = rng.permutation(pnls)
        kap = kapital
        for j, p in enumerate(sh):
            kap *= (1 + p / 100)
            mat[s, j+1] = kap
    return mat

print('Backtest engine naložen ✓')

Backtest engine naložen ✓


---
## 3 · Vizualizacija

In [5]:
def narisi_graf(trades, equity, mc_matrix, stats, kapital, interval, n_sim):
    if not trades:
        print('Ni trade-ov za prikaz.')
        return

    BG    = '#0d0d0d'
    PANEL = '#161616'
    GRID  = '#252525'
    BLUE  = '#4a9eff'
    GREEN = '#4caf50'
    RED   = '#f44336'
    AMBER = '#ffb347'
    MUTED = '#777777'
    WHITE = '#e0e0e0'

    fig = plt.figure(figsize=(17, 14), facecolor=BG)
    fig.suptitle(
        f'BTC/USD  ·  Backtest + Monte Carlo  ·  TF: {interval}  ·  '
        f'{len(trades)} trades  ·  začetni kapital: {kapital:.0f} EUR',
        color=WHITE, fontsize=12, fontweight='bold', y=0.985
    )

    gs = gridspec.GridSpec(3, 3, figure=fig,
                           hspace=0.44, wspace=0.33,
                           left=0.06, right=0.97, top=0.945, bottom=0.06)

    def style_ax(ax, title=''):
        ax.set_facecolor(PANEL)
        ax.tick_params(colors=MUTED, labelsize=8)
        for sp in ax.spines.values():
            sp.set_edgecolor(GRID)
        ax.grid(color=GRID, linewidth=0.4, alpha=0.7)
        if title:
            ax.set_title(title, color=WHITE, fontsize=9, pad=6)

    pnls = np.array([t['pnl_pct'] for t in trades])
    x_eq = np.arange(len(equity))

    # ── 1. Equity + MC ──────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    style_ax(ax1, 'Equity krivulja + Monte Carlo simulacije')

    n_draw = min(n_sim, mc_matrix.shape[0])
    x_mc   = np.arange(mc_matrix.shape[1])
    for s in range(n_draw):
        ax1.plot(mc_matrix[s], color=GREEN, alpha=0.035, linewidth=0.6)

    p5   = np.percentile(mc_matrix, 5,  axis=0)
    p95  = np.percentile(mc_matrix, 95, axis=0)
    med  = np.percentile(mc_matrix, 50, axis=0)
    ax1.fill_between(x_mc, p5, p95, color=GREEN, alpha=0.10, label='MC 5–95 percentil')
    ax1.plot(x_mc, med, color=GREEN, lw=1.2, alpha=0.6, linestyle='--', label='MC median')
    ax1.plot(x_eq, equity, color=BLUE, lw=2.2, zorder=5, label='Dejanska equity')
    ax1.axhline(kapital, color='#555', lw=0.7, linestyle=':', label=f'Start ({kapital:.0f} EUR)')

    for idx in range(len(trades)):
        col = GREEN if trades[idx]['pnl_pct'] > 0 else RED
        ax1.axvline(idx + 1, color=col, alpha=0.08, linewidth=0.5)

    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax1.set_ylabel('Kapital (EUR)', color=MUTED, fontsize=8)
    ax1.set_xlabel('Trade #', color=MUTED, fontsize=8)
    ax1.legend(fontsize=7.5, loc='upper left',
               facecolor='#1a1a1a', edgecolor=GRID, labelcolor=WHITE)

    # ── 2. Drawdown ──────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, :2])
    style_ax(ax2, 'Drawdown (%)')
    dd = calc_drawdown(equity)
    ax2.fill_between(x_eq, dd, 0, color=RED, alpha=0.30)
    ax2.plot(x_eq, dd, color=RED, lw=1.0)
    ax2.axhline(stats.get('max_dd_pct', 0), color=AMBER, lw=0.8, linestyle='--', alpha=0.8)
    ax2.text(0.5, stats.get('max_dd_pct', 0) + 0.2,
             f"max DD: {stats.get('max_dd_pct',0):.1f}%",
             color=AMBER, fontsize=7.5)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}%'))
    ax2.set_ylabel('%', color=MUTED, fontsize=8)
    ax2.set_xlabel('Trade #', color=MUTED, fontsize=8)

    # ── 3. P&L distribucija  (POPRAVEK: barve kot array, ne list) ────────
    ax3 = fig.add_subplot(gs[1, 2])
    style_ax(ax3, 'P&L distribucija')
    n_bins  = 28
    counts, bin_edges = np.histogram(pnls, bins=n_bins)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bar_colors = np.where(bin_centers >= 0, GREEN, RED)  # numpy array — ne list
    ax3.bar(bin_centers, counts,
            width=(bin_edges[1] - bin_edges[0]) * 0.85,
            color=bar_colors, edgecolor=PANEL, linewidth=0.3)
    ax3.axvline(0, color=WHITE, lw=0.8, alpha=0.5)
    ax3.axvline(float(np.mean(pnls)), color=AMBER, lw=1.0, linestyle='--',
                label=f'avg {np.mean(pnls):.1f}%')
    ax3.set_xlabel('P&L %', color=MUTED, fontsize=8)
    ax3.set_ylabel('Trades', color=MUTED, fontsize=8)
    ax3.legend(fontsize=7, facecolor='#1a1a1a', edgecolor=GRID, labelcolor=WHITE)

    # ── 4. Statistike ────────────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[2, 0])
    style_ax(ax4, 'Ključne statistike')
    ax4.axis('off')
    donos_col = GREEN if stats.get('skupni_donos_pct', 0) >= 0 else RED
    rows = [
        ('Skupaj trades',  str(stats.get('skupaj_trades', 0)),         WHITE),
        ('  Long',         str(stats.get('long_trades', 0)),           MUTED),
        ('  Short',        str(stats.get('short_trades', 0)),          MUTED),
        ('Win rate',       f"{stats.get('win_rate',0):.1f}%",
             GREEN if stats.get('win_rate', 0) >= 50 else RED),
        ('TP / SL',        f"{stats.get('tp_trades',0)} / {stats.get('sl_trades',0)}", WHITE),
        ('Avg win',        f"+{stats.get('avg_win',0):.2f}%",          GREEN),
        ('Avg loss',       f"{stats.get('avg_loss',0):.2f}%",         RED),
        ('Avg R:R',        f"1 : {stats.get('rr_avg',0):.2f}",        WHITE),
        ('Sharpe',         f"{stats.get('sharpe',0):.2f}",             WHITE),
        ('Max DD',         f"{stats.get('max_dd_pct',0):.1f}%",       RED),
        ('Donos',          f"{stats.get('skupni_donos_pct',0):+.1f}%", donos_col),
        ('Končni kapital', f"{stats.get('konc_kapital',0):.2f} EUR",  donos_col),
        ('Zaslužek',       f"{stats.get('skupni_donos_eur',0):+.2f} EUR", donos_col),
    ]
    for row_i, (k, v, vc) in enumerate(rows):
        y = 1 - row_i * 0.077
        ax4.text(0.02, y, k + ':', color=MUTED,  fontsize=8.0, transform=ax4.transAxes, va='top')
        ax4.text(0.62, y, v,       color=vc,     fontsize=8.0, fontweight='bold',
                 transform=ax4.transAxes, va='top')

    # ── 5. MC końni kapital distribucija ─────────────────────────────────
    ax5 = fig.add_subplot(gs[2, 1])
    final_caps  = mc_matrix[:, -1]
    pct_prof    = np.sum(final_caps > kapital) / len(final_caps) * 100
    style_ax(ax5, f'MC — končni kapital  ({pct_prof:.0f}% v dobičku)')

    counts_mc, edges_mc = np.histogram(final_caps, bins=30)
    centers_mc  = (edges_mc[:-1] + edges_mc[1:]) / 2
    colors_mc   = np.where(centers_mc >= kapital, GREEN, RED)
    ax5.bar(centers_mc, counts_mc,
            width=(edges_mc[1] - edges_mc[0]) * 0.85,
            color=colors_mc, edgecolor=PANEL, linewidth=0.3)
    ax5.axvline(kapital,                       color=WHITE, lw=0.8, linestyle=':',
                label=f'Start ({kapital:.0f})')
    ax5.axvline(np.median(final_caps),         color=AMBER, lw=1.0, linestyle='--',
                label=f'Mediana ({np.median(final_caps):.0f})')
    ax5.axvline(np.percentile(final_caps, 5),  color=RED,   lw=0.8, linestyle='--',
                alpha=0.8, label=f'5p ({np.percentile(final_caps,5):.0f})')
    ax5.axvline(np.percentile(final_caps, 95), color=GREEN, lw=0.8, linestyle='--',
                alpha=0.8, label=f'95p ({np.percentile(final_caps,95):.0f})')
    ax5.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax5.set_xlabel('EUR', color=MUTED, fontsize=8)
    ax5.set_ylabel('Simulacije', color=MUTED, fontsize=8)
    ax5.legend(fontsize=7, facecolor='#1a1a1a', edgecolor=GRID, labelcolor=WHITE)

    # ── 6. Trade P&L bar chart ────────────────────────────────────────────
    ax6 = fig.add_subplot(gs[2, 2])
    style_ax(ax6, 'P&L po trade-ih')
    idxs       = np.arange(len(pnls))
    bar_cols   = np.where(pnls > 0, GREEN, RED)
    ax6.bar(idxs, pnls, color=bar_cols, width=0.8, edgecolor='none')
    ax6.axhline(0, color=WHITE, lw=0.6, alpha=0.5)
    ax6.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}%'))
    ax6.set_xlabel('Trade #', color=MUTED, fontsize=8)
    ax6.set_ylabel('P&L %',   color=MUTED, fontsize=8)
    leg_els = [
        mpatches.Patch(color=GREEN, label='Dobiček'),
        mpatches.Patch(color=RED,   label='Izguba'),
    ]
    ax6.legend(handles=leg_els, fontsize=7,
               facecolor='#1a1a1a', edgecolor=GRID, labelcolor=WHITE)

    plt.savefig('btc_backtest_rezultat.png', dpi=130, bbox_inches='tight', facecolor=BG)
    plt.show()
    print('Graf shranjen: btc_backtest_rezultat.png')

print('Vizualizacija naložena ✓')

Vizualizacija naložena ✓


---
## 4 · Interaktivni widget — nastavi parametre in poženi

In [6]:
# ─── CACHE: podatki se naložijo enkrat in se shranijo ────────────────────────
_df_cache = {}

def get_data(interval, period):
    key = f'{interval}_{period}'
    if key not in _df_cache:
        print(f'  Nalagam BTC-USD ({interval}, {period})...')
        df = yf.Ticker('BTC-USD').history(interval=interval, period=period)
        df = df.dropna(subset=['Close', 'High', 'Low'])
        _df_cache[key] = df
        print(f'  Naloženo: {len(df)} svečk')
    return _df_cache[key]

# ─── WIDGETI ─────────────────────────────────────────────────────────────────
w_tf = widgets.ToggleButtons(
    options=[('1h — 2 leti', '1h_730d'), ('1d — 2 leti', '1d_730d'),
             ('1h — 1 leto', '1h_365d'), ('1d — 3 leta', '1d_1095d')],
    value='1h_730d',
    description='Timeframe:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%')
)

w_kapital = widgets.FloatText(
    value=1000.0, min=10.0, step=100.0,
    description='Kapital (EUR):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)

w_long_prag = widgets.IntSlider(
    value=3, min=2, max=6, step=1,
    description='Long prag (tocke ≥):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px')
)

w_short_prag = widgets.IntSlider(
    value=-3, min=-6, max=-2, step=1,
    description='Short prag (tocke ≤):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px')
)

w_sl = widgets.FloatSlider(
    value=2.5, min=0.5, max=5.0, step=0.25,
    description='SL mult (ATR ×):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px'),
    readout_format='.2f'
)

w_tp = widgets.FloatSlider(
    value=5.0, min=1.0, max=10.0, step=0.25,
    description='TP mult (ATR ×):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px'),
    readout_format='.2f'
)

w_provizija = widgets.FloatText(
    value=0.06, min=0.0, max=1.0, step=0.01,
    description='Provizija %:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)

w_n_sim = widgets.IntSlider(
    value=200, min=50, max=2000, step=50,
    description='MC simulacij:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px')
)

w_seed = widgets.IntText(
    value=42, description='MC seed:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)

w_btn = widgets.Button(
    description=' Poženi backtest',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='200px', height='40px')
)

out = widgets.Output()

# ─── LAYOUT ──────────────────────────────────────────────────────────────────
row1 = widgets.HBox([w_kapital, w_provizija, w_seed])
row2 = widgets.HBox([w_long_prag, w_short_prag])
row3 = widgets.HBox([w_sl, w_tp])
row4 = widgets.HBox([w_n_sim])

ui = widgets.VBox([
    widgets.HTML('<h3 style="margin:8px 0 4px">BTC Backtest — parametri</h3>'),
    w_tf,
    widgets.HTML('<hr style="margin:6px 0">'),
    row1,
    widgets.HTML('<b>Signal prag</b>'),
    row2,
    widgets.HTML('<b>SL / TP multiplikatorji</b>'),
    row3,
    widgets.HTML('<b>Monte Carlo</b>'),
    row4,
    widgets.HTML('<hr style="margin:6px 0">'),
    w_btn,
])

# ─── CALLBACK ────────────────────────────────────────────────────────────────
def pozeni(_):
    with out:
        clear_output(wait=True)

        tf_val    = w_tf.value                # npr. '1h_730d'
        interval  = tf_val.split('_')[0]      # '1h'
        period    = tf_val.split('_')[1]      # '730d'
        kapital   = float(w_kapital.value)
        provizija = float(w_provizija.value) / 100.0
        sig_long  = int(w_long_prag.value)
        sig_short = int(w_short_prag.value)
        sl_m      = float(w_sl.value)
        tp_m      = float(w_tp.value)
        n_sim     = int(w_n_sim.value)
        seed      = int(w_seed.value)

        print(f'Nalagam podatke...')
        try:
            df = get_data(interval, period)
        except Exception as e:
            print(f'NAPAKA pri pridobivanju podatkov: {e}')
            return

        print(f'Izvajam backtest  (long≥{sig_long}, short≤{sig_short}, SL={sl_m}×, TP={tp_m}×)...')
        trades = run_backtest(
            df, interval=interval,
            signal_long=sig_long, signal_short=sig_short,
            sl_mult=sl_m, tp_mult=tp_m,
            provizija=provizija
        )

        if not trades:
            print('Ni signalov! Poskusi z nižjim pragom (npr. long≥2).')
            return

        print(f'  Najdeno {len(trades)} trade-ov')

        equity = equity_krivulja(trades, kapital)
        stats  = calc_statistike(trades, equity, kapital)

        print(f'Izvajam {n_sim} MC simulacij...')
        mc = monte_carlo(trades, n_sim=n_sim, kapital=kapital, seed=seed)

        # ── Hitra statistika v besedilu ──────────────────────────────────
        final = mc[:, -1]
        print(f"\n{'─'*52}")
        print(f"  Trades          : {stats['skupaj_trades']}  (L:{stats['long_trades']} S:{stats['short_trades']})")
        print(f"  Win rate        : {stats['win_rate']:.1f}%")
        print(f"  TP / SL         : {stats['tp_trades']} / {stats['sl_trades']}")
        print(f"  Avg win / loss  : +{stats['avg_win']:.2f}% / {stats['avg_loss']:.2f}%")
        print(f"  Sharpe          : {stats['sharpe']:.2f}")
        print(f"  Max DD          : {stats['max_dd_pct']:.1f}%")
        print(f"  Skupni donos    : {stats['skupni_donos_pct']:+.1f}%")
        print(f"  Končni kapital  : {stats['konc_kapital']:.2f} EUR")
        print(f"  MC mediana      : {np.median(final):,.2f} EUR")
        print(f"  MC 5–95 perc.   : {np.percentile(final,5):,.0f} – {np.percentile(final,95):,.0f} EUR")
        print(f"  MC % v dobičku  : {np.sum(final > kapital)/len(final)*100:.1f}%")
        print(f"{'─'*52}\n")

        narisi_graf(trades, equity, mc, stats, kapital, interval, n_sim)

w_btn.on_click(pozeni)

display(ui, out)
print('Widget pripravljen — nastavi parametre in klikni Poženi backtest')

Output()

Widget pripravljen — nastavi parametre in klikni Poženi backtest


---
## 5 · Tabela zadnjih trade-ov (po zagonu)

In [ ]:
# Poženi to celico po zagonu backtesta zgoraj
# Da dostop do zadnjih rezultatov za nadaljnjo analizo

try:
    import pandas as pd

    tf_val   = w_tf.value
    interval = tf_val.split('_')[0]
    period   = tf_val.split('_')[1]
    kapital  = float(w_kapital.value)
    df       = _df_cache.get(f'{interval}_{period}')

    if df is None:
        print('Najprej poženi backtest v celici 4.')
    else:
        trades_df = run_backtest(
            df, interval=interval,
            signal_long  = int(w_long_prag.value),
            signal_short = int(w_short_prag.value),
            sl_mult      = float(w_sl.value),
            tp_mult      = float(w_tp.value),
            provizija    = float(w_provizija.value) / 100.0
        )

        tbl = pd.DataFrame(trades_df)[[
            'vstop_datum', 'smer', 'vstop_cena', 'izhod_cena',
            'tocke', 'pnl_pct', 'izhod_vzrok', 'atr', 'sigma'
        ]].tail(30).copy()

        tbl['vstop_datum'] = pd.to_datetime(tbl['vstop_datum']).dt.strftime('%Y-%m-%d %H:%M')
        tbl['vstop_cena']  = tbl['vstop_cena'].map('{:,.0f}'.format)
        tbl['izhod_cena']  = tbl['izhod_cena'].map('{:,.0f}'.format)
        tbl['pnl_pct']     = tbl['pnl_pct'].map('{:+.2f}%'.format)
        tbl['atr']         = tbl['atr'].map('{:,.0f}'.format)
        tbl['sigma']       = tbl['sigma'].map('{:.1%}'.format)
        tbl.columns        = ['Datum vstopa','Smer','Vstop ($)','Izhod ($)',
                               'Tocke','P&L','Izhod vzrok','ATR','Vol (RH)']
        tbl = tbl.reset_index(drop=True)
        tbl.index += 1

        def barvaj(row):
            c = '#1a3a1a' if '+' in str(row['P&L']) else '#3a1a1a'
            return [f'background-color: {c}'] * len(row)

        display(tbl.style
            .apply(barvaj, axis=1)
            .set_properties(**{'font-size': '11px', 'color': '#e0e0e0',
                               'background-color': '#111'})
            .set_table_styles([{
                'selector': 'th',
                'props': [('background-color', '#1e1e1e'), ('color', '#aaa'),
                          ('font-size', '11px'), ('border', '1px solid #333')]
            }])
        )
        print(f'Prikazanih zadnjih {len(tbl)} trade-ov')

except Exception as e:
    print(f'Napaka: {e}')